Linear Algebra Fundamentals

1. Build Vector From Scratch

In [25]:
class Vector:
    def __init__(self, components:List):
        self.components = components
        self.dim = len(self.components)
    def __add__(self,other:Vector):
        return Vector([a+b for a,b in zip(self.components,other.components)])
        
    def __sub__(self,other):
        return Vector([a-b for a,b in zip(self.components,other.components)])

    def dot(self,other):
         return sum([a*b for a,b in zip(self.components,other.components)])

    def magnitude(self):
        return sum(i**2 for i in self.components)**0.5

    def normalize(self):
        magnitude = self.magnitude()
        return Vector([x/magnitude for x in self.components])

    def cosine_similarity(self,other):
        dot_product = self.dot(other)
        magnitude_self = self.magnitude()
        magnitude_other = other.magnitude()
        return dot_product/(magnitude_other*magnitude_self)

    def __repr__(self):
        return f"Vector({self.components})"

Testing

In [26]:
a = Vector([1,2,3])
b= Vector([2,3,4])
print(f"a + b = {a + b}")
print(f"a · b = {a.dot(b)}")
print(f"|a| = {a.magnitude():.4f}")
print(f"cosine similarity = {a.cosine_similarity(b):.4f}")

a + b = Vector([3, 5, 7])
a · b = 20
|a| = 3.7417
cosine similarity = 0.9926


2. Matrix from scratch

In [36]:
class Matrix:
    def __init__(self,rows):
        self.rows = [list(row) for row in rows]
        self.shape = [len(self.rows),len(self.rows[0])]
    def __matmul__(self,other):
        if isinstance(other,Vector):
            return Vector([
                sum(self.rows[i][j]* other.components[j] for j in range(self.shape[1]))
                    for i in range(self.shape[0])
            ])
        rows = []
        for i in range(self.shape[0]):
            row = []
            for j in range(other.shape[1]):
                    row.append(sum(
                        self.row[i][k]*other.rows[k][j]
                        for k in range(self.shape[1])
                    ))
            rows.append(row)
        return Matrix(rows)
    def transpose(self):
        return Matrix([
            [self.rows[j][i] for j in range(self.shape[0])]
            for i in range(self.shape[1])
        ])     
    def __repr__(self):
        return f"Matrix ({self.rows})"
        
rotation_90 = Matrix([[0, -1], [1, 0]])
point = Vector([3, 1])

rotated = rotation_90 @ point
print(f"Original: {point}")
print(f"Rotated 90°: {rotated}")

Original: Vector([3, 1])
Rotated 90°: Vector([-1, 3])


Step 3: Why this matters for AI

In [38]:
import random
random.seed(42)
weights = Matrix([[random.gauss(0,0.1) for _ in range(3)] for _ in range(2)])
input_vector = Vector([1.0,0.5,-0.3])

output = weights @ input_vector
print(f"Input (3D): {input_vector}")
print(f"Output (2D): {output}")
print("This is what a neural network layer does -- matrix multiplication.")

Input (3D): Vector([1.0, 0.5, -0.3])
Output (2D): Vector([-0.019714737127338927, 0.10873956075097067])
This is what a neural network layer does -- matrix multiplication.


In [42]:
def is_linearly_independent(vectors):
    n = len(vectors)
    dim = len(vectors[0].components)
    mat = Matrix([v.components[:] for v in vectors])
    rows = [row[:] for row in mat.rows]
    rank = 0
    for col in range(dim):
        pivot = None
        for row in range(rank,len(rows)):
            if abs(rows[row][col]) > 1e-10:
                pivot = row
                break
            if pivot is None:
                continue
            rows[rank],ros[privot] = rows[pivot],rows[rank]
            scale = rows[rank][col]
            rows[rank] = [x/scale for x in rows[rank]]
            for row in range(len(rows)):
                if row != rank and abs(rows[row][col]) > 1e-10:
                    factor = rows[row][col]
                    rows[row] = [rows[row][j] -factor*rows[rank][j] for j in range(dim)]
            rank += 1
    return rank == n

def project(a,b):
    scalar = a.dot(b)/b.dot(b)
    return Vector([scalar * x for x in b.components])

def gram_schmidt(vectors):
    orthonormal = []
    for v in vectors:
        w=v
        for u in orthonormal:
            proj = project(w,u)
            w = w - proj

        if w.magnitude() <1e-10:
            continue
        orthonormal.append(w.normalize())
    return orthonormal
v1 = Vector([1, 0, 0])
v2 = Vector([1, 1, 0])
v3 = Vector([1, 1, 1])
basis = gram_schmidt([v1, v2, v3])
for i, u in enumerate(basis):
    print(f"u{i+1} = {u}")
    print(f"  |u{i+1}| = {u.magnitude():.6f}")

print(f"u1 · u2 = {basis[0].dot(basis[1]):.6f}")
print(f"u1 · u3 = {basis[0].dot(basis[2]):.6f}")
print(f"u2 · u3 = {basis[1].dot(basis[2]):.6f}")       

u1 = Vector([1.0, 0.0, 0.0])
  |u1| = 1.000000
u2 = Vector([0.0, 1.0, 0.0])
  |u2| = 1.000000
u3 = Vector([0.0, 0.0, 1.0])
  |u3| = 1.000000
u1 · u2 = 0.000000
u1 · u3 = 0.000000
u2 · u3 = 0.000000


Exercises:

1. Implement Vector.angle_between(other) that returns the angle in degrees between two vectors
2. Create a 2D scaling matrix that doubles the x-coordinate and triples the y-coordinate, then apply it to the vector [1, 1]
3. Given 5 random word-like vectors (dimension 50), find the two most similar using cosine similarity
4. Verify that the Gram-Schmidt output is truly orthonormal: check that every pair has dot product 0 and every vector has magnitude 1
5. Create a 3x3 matrix with rank 2. Verify using the rank() method. Then explain what geometric object the columns span.
6. Project the vector [1, 2, 3] onto [1, 1, 1]. What does the result represent geometrically?